In [1]:
import numpy as np

N_AL = 50  # number of Antennal Lobe neurons
N_ODOR_ACTIVE = 20  # how many AL neurons each odor activates

def make_odor(n_al=N_AL, n_active=N_ODOR_ACTIVE, seed=None):
    """
    Returns a binary vector of length n_al
    with n_active neurons set to 1.
    Each odor activates a random non-overlapping subset.
    """
    rng = np.random.default_rng(seed)
    odor = np.zeros(n_al)
    active_indices = rng.choice(n_al, size=n_active, replace=False)
    odor[active_indices] = 1.0
    return odor

# Create two non-overlapping odors
odor1 = make_odor(seed=42)
odor2 = make_odor(seed=99)

# Students could verify they don't overlap:
# assert np.dot(odor1, odor2) == 0

In [2]:
import numpy as np

def sigmoid(x, gain=10, threshold=0.5):
    """
    Squashes input into 0-1 range.
    gain controls steepness,
    threshold controls the midpoint.
    Students can experiment with these values.
    """
    return 1 / (1 + np.exp(-gain * (x - threshold)))

class SimulatedFly:
    def __init__(self, n_al=50, n_mb=200, seed=0):
        """
        Reduced size from paper's 2000 MB neurons
        to 200 for speed — results should be qualitatively similar.
        Students could test whether this holds.
        """
        rng = np.random.default_rng(seed)

        self.n_al = n_al
        self.n_mb = n_mb

        # --- Connection weight matrices ---
        # AL -> MB: sparse random connections
        # Each MB neuron connects to ~20 AL neurons on average
        # (matching paper's optimal connectivity)
        self.w_al_mb = rng.random((n_mb, n_al)) * 0.1

        # MB -> Output: single output neuron
        # represented as a vector of length n_mb
        self.w_mb_out = rng.random(n_mb) * 0.1

        # Store activity history for plotting
        self.output_history = []
        self.w_mb_out_history = []

    def forward(self, odor_pattern, dopamine=0.0):
        """
        One forward pass through the network.
        Returns output neuron activity.

        odor_pattern: binary vector of length n_al
        dopamine: float, 1.0 if shock present, 0.0 if not
        """
        # AL -> MB
        mb_input = self.w_al_mb @ odor_pattern
        mb_activity = sigmoid(mb_input)

        # Apply inhibition: suppresses weakly activated MB neurons
        # Simple approximation of paper's inhibitory neuron
        inhibition_threshold = np.percentile(mb_activity, 70)
        mb_activity[mb_activity < inhibition_threshold] = 0.0

        # MB -> Output
        out_input = np.dot(self.w_mb_out, mb_activity)
        out_activity = sigmoid(out_input)

        # Store for later
        self.output_history.append(out_activity)

        return mb_activity, out_activity

    def avoidance(self, out_activity, threshold=0.7):
        """
        Does the fly avoid the odor?
        Threshold is a hyperparameter students can adjust.
        """
        return out_activity > threshold

In [3]:
import numpy as np

def hebbian_update(w, pre_activity, post_activity,
                   dopamine, learning_rate=0.01):
    """
    Updates MB -> Output weights.

    Strengthens connection if:
    - both pre and post neurons are active (Hebbian)
    - dopamine is present (shock occurred)

    Weakens connection if:
    - pre neuron active but no dopamine (extinction)

    This combines equations 8, 9, 10 from the paper
    into a single simplified rule.
    """
    if dopamine > 0:
        # Odor + shock: strengthen weights
        dw = learning_rate * pre_activity * post_activity * dopamine
    else:
        # Odor alone: slight weakening (extinction)
        dw = -learning_rate * 0.1 * pre_activity * post_activity

    w += dw

    # Keep weights in a sensible range
    w = np.clip(w, 0, 1)
    return w


def retrograde_update(w_al_mb, mb_activity, out_activity,
                      learning_rate=0.005):
    """
    Simplified retrograde signaling.

    When output neuron is highly active, nearby MB neuron
    input synapses get strengthened — approximating the
    nitric oxide diffusion in the paper.

    This is the most simplified part of the model.
    Students could make this more realistic as an extension.
    """
    if out_activity > 0.5:
        # Active MB neurons get their input weights boosted
        # proportional to both their own activity and
        # the output neuron's activity
        for i, activity in enumerate(mb_activity):
            if activity > 0:
                dw = learning_rate * activity * out_activity
                w_al_mb[i] += dw

    w_al_mb = np.clip(w_al_mb, 0, 1)
    return w_al_mb

In [4]:
from network import SimulatedFly
from stimuli import make_odor
from learning import hebbian_update, retrograde_update
from plotting import plot_results
import numpy as np

# --- Setup ---
fly = SimulatedFly(n_al=50, n_mb=200)

odor1 = make_odor(seed=42)
odor2 = make_odor(seed=99)

N_TRIALS_PHASE1 = 10   # odor1 + shock
N_TRIALS_PHASE2 = 10   # odor1 + odor2
N_TRIALS_PHASE3 = 5    # odor2 alone (test)

output_history = {"phase1": [], "phase2": [], "phase3": []}

# --- Phase 1: First order conditioning ---
# Pair odor1 with shock
print("Phase 1: odor1 + shock")
for trial in range(N_TRIALS_PHASE1):
    mb_activity, out_activity = fly.forward(odor1, dopamine=1.0)

    # Update weights
    fly.w_mb_out = hebbian_update(
        fly.w_mb_out, mb_activity, out_activity, dopamine=1.0
    )
    fly.w_al_mb = retrograde_update(
        fly.w_al_mb, mb_activity, out_activity
    )

    output_history["phase1"].append(out_activity)
    print(f"  Trial {trial+1}: output={out_activity:.3f}, "
          f"avoidance={fly.avoidance(out_activity)}")

# --- Phase 2: Second order conditioning ---
# Pair odor1 with odor2 (no shock)
print("\nPhase 2: odor1 + odor2")
for trial in range(N_TRIALS_PHASE2):
    # Present both odors simultaneously
    combined_odor = np.clip(odor1 + odor2, 0, 1)
    mb_activity, out_activity = fly.forward(combined_odor, dopamine=0.0)

    fly.w_mb_out = hebbian_update(
        fly.w_mb_out, mb_activity, out_activity, dopamine=0.0
    )
    fly.w_al_mb = retrograde_update(
        fly.w_al_mb, mb_activity, out_activity
    )

    output_history["phase2"].append(out_activity)
    print(f"  Trial {trial+1}: output={out_activity:.3f}, "
          f"avoidance={fly.avoidance(out_activity)}")

# --- Phase 3: Test odor2 alone ---
print("\nPhase 3: odor2 alone (test)")
for trial in range(N_TRIALS_PHASE3):
    mb_activity, out_activity = fly.forward(odor2, dopamine=0.0)

    output_history["phase3"].append(out_activity)
    print(f"  Trial {trial+1}: output={out_activity:.3f}, "
          f"avoidance={fly.avoidance(out_activity)}")

plot_results(output_history)

ModuleNotFoundError: No module named 'network'

In [ ]:
import matplotlib.pyplot as plt

def plot_results(output_history):
    """
    Plots output neuron activity across all three phases.
    The key result to look for:
    - Phase 1: activity should rise above avoidance threshold
    - Phase 2: activity may dip then stabilise
    - Phase 3: activity should still be above threshold
      (demonstrating second order conditioning worked)
    """
    fig, ax = plt.subplots(figsize=(10, 4))

    # Concatenate all phases for continuous x axis
    all_outputs = (output_history["phase1"] +
                   output_history["phase2"] +
                   output_history["phase3"])

    n1 = len(output_history["phase1"])
    n2 = n1 + len(output_history["phase2"])

    ax.plot(all_outputs, color="steelblue", linewidth=2)

    # Mark phase boundaries
    ax.axvline(x=n1, color="gray", linestyle="--", label="Phase 2 starts")
    ax.axvline(x=n2, color="gray", linestyle=":",  label="Phase 3 starts")

    # Mark avoidance threshold
    ax.axhline(y=0.7, color="red", linestyle="-.",
               linewidth=1, label="Avoidance threshold")

    ax.set_xlabel("Trial")
    ax.set_ylabel("Output Neuron Activity")
    ax.set_title("Simulated Fly Conditioning")
    ax.legend()
    plt.tight_layout()
    plt.savefig("conditioning_results.png", dpi=150)
    plt.show()